# Pretrain on unlabeled data

## Evaluating generative text models

### Using the GPT to generate text

In [5]:
import torch

In [6]:
from importlib.metadata import version

pkgs = [
    "matplotlib",
    "numpy",
    "torch",
    "tiktoken",
    # "tensorflow" 
]

for p in pkgs:
    print(f"{p} version:{version(p)}")

matplotlib version:3.10.8
numpy version:2.4.2
torch version:2.12.0
tiktoken version:0.13.0


In [32]:
from prev_chp import GPTModel

GPT_CONFIG_124M = {
    "n_layers": 12,
    "n_heads": 12,
    "emb_dim": 768,
    "vocab_size": 27,
    "context_length": 256 ,
    "drop_rate": 0.1,
    "qkv_bias": False,
}



In [33]:
torch.manual_seed(123)

model = GPTModel(GPT_CONFIG_124M)
model.eval();

In [34]:
import tiktoken

# def text_to_token_ids(text, tokenizer):
#     encode = tokenizer.encode(text, allowed_special="<|endoftext|>")
#     encode_tensor = torch.tensor(encode).unsqueeze(0) # add the batch dimension 
#     return encode_tensor


In [35]:
text = open("../names.txt", "r").read()
chars = sorted(list(set(text)))
chars[0] = " "
len(chars)

27

In [36]:
stoi = {s:i for i, s in enumerate(chars)}
itos = {i:s for i, s in enumerate(chars)}

def text_to_token_ids(sen):
    encode =[stoi[s] for s in sen]
    encode_tensor = torch.tensor(encode).unsqueeze(0)
    return encode_tensor

def token_ids_to_text(ids):
    arr = ids.squeeze(0).tolist()
    return  ''.join([itos[s] for s in arr])


In [37]:
text = "Khalid".lower()

# tokenizer = tiktoken.get_encoding('gpt2')
# text_to_token_ids(text, encode)


# replace this with actual gpt2 encoder

token_ids = text_to_token_ids(text)
# decode(token_ids)

token_ids.shape, token_ids_to_text(token_ids)

(torch.Size([1, 6]), 'khalid')

In [38]:
(token_ids.squeeze(0)).tolist()

[11, 8, 1, 12, 9, 4]

In [41]:
def generate_text_simple(model, idx, max_new_tokens, context_size):
    # idx is (batch, n_tokens) array of indices in the current context
    for _ in range(max_new_tokens):
        
        # Crop current context if it exceeds the supported context size
        # E.g., if LLM supports only 5 tokens, and the context size is 10
        # then only the last 5 tokens are used as context
        idx_cond = idx[:, -context_size:]
        # print(f"idx conde {idx_cond} {idx_cond.shape}")
        # Get the predictions
        with torch.no_grad():
            logits = model(idx_cond)
        
        # print(f"logits {logits} {logits.shape}")

        # Focus only on the last time step
        # (batch, n_tokens, vocab_size) becomes (batch, vocab_size)
        logits = logits[:, -1, :]  

        # print(f"logits {logits} {logits.shape}")

        # Apply softmax to get probabilities
        probas = torch.softmax(logits, dim=-1)  # (batch, vocab_size)
        # print(f"probas {probas} {probas.shape}")

        # Get the idx of the vocab entry with the highest probability value
        idx_next = torch.argmax(probas, dim=-1, keepdim=True)  # (batch, 1)
        # print(f"idx next {idx_next} {probas.shape}")

        # Append sampled index to the running sequence
        idx = torch.cat((idx, idx_next), dim=1)  # (batch, n_tokens+1)

    return idx

In [46]:
predications = generate_text_simple(
    model=model,
    idx=token_ids,
    max_new_tokens=100,
    context_size=GPT_CONFIG_124M['context_length']
)

In [47]:
token_ids_to_text(predications)

'khalidikpvleppvppbfsplpbawuspslwfek wleulvtlegalwfwlvqdqpppidwgekvhwo pspphpsavavvpvvvvvvvqppsawfwopblvpvp'